# Pre-training



---



Pre-training:
> Learning from massive, unlabeled datasets (the "Internet-scale" knowledge)

Sochiye ek bachha hai jise bhasha (language) nahi aati. Aap usse duniya ki sabse badi library mein chhod dete hain jahan internet ki saari books, articles, aur Wikipedia maujood hai.

* Bachha kisi teacher se nahi seekhta.

* Wo bas un kitabon ko padhta rehta hai aur pattern notice karta hai.

* Use samajh aane lagta hai ki "Chai" ke baad aksar "pina" word aata hai, aur "Newton" ke saath "Gravity".

**Technical Definition:**

Pre-training ek Self-Supervised Learning process hai jahan model ko kisi human label (jaise "ye cat hai") ki zaroorat nahi hoti. Wo raw text se khud patterns seekhta hai.

## 2. Massive & Unlabeled Datasets (The Data Scale)
"Internet-scale" ka matlab hai itna data jo ek insaan hazaron saalon mein bhi nahi padh sakta. Isme kya-kya hota hai?

* Common Crawl: Poore internet ka ek bada hissa.

* Wikipedia: Facts aur knowledge ke liye.

* Books (Gutenberg/Books3): Kahaniyaan aur gehri bhasha samajhne ke liye.

* GitHub Code: Logic aur programming seekhne ke liye.

## 3. Model Seekhta Kya Hai? (The Objective)
Pre-training ke waqt model do main kaam (tasks) karta hai:

1. **Next Token Prediction (GPT Style)**

    Model ko ek adhoora sentence diya jata hai: "The sky is ____"
    Model guess karta hai: "Blue".
    Agar wo galat guess kare (e.g., "Green"), to wo apne Weights ko adjust karta hai taaki agli baar sahi guess kare.

2. **Masked Language Modeling (BERT Style)**
    Beech ka word chhupa diya jata hai: "I want to [MASK] some water."
    Model ko aas-paas ke words dekh kar samajhna hota hai ki wahan "drink" aayega.

## 4. Practical Implementation

In [ ]:
from transformers import pipeline

In [ ]:
fill_mask = pipeline('fill-mask',model = 'bert-base-uncased')
result = fill_mask('The president of USA lives in the [MASK] House.')

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
result

[{'score': 0.1772841513156891,
  'token': 2317,
  'token_str': 'white',
  'sequence': 'the president of usa lives in the white house.'},
 {'score': 0.0885426327586174,
  'token': 4113,
  'token_str': 'guest',
  'sequence': 'the president of usa lives in the guest house.'},
 {'score': 0.06242973729968071,
  'token': 2168,
  'token_str': 'same',
  'sequence': 'the president of usa lives in the same house.'},
 {'score': 0.03342711180448532,
  'token': 4883,
  'token_str': 'presidential',
  'sequence': 'the president of usa lives in the presidential house.'},
 {'score': 0.026277145370841026,
  'token': 3221,
  'token_str': 'glass',
  'sequence': 'the president of usa lives in the glass house.'}]

In [ ]:
for i,j in enumerate(result):
  print(f'pred {i+1}: {j['token_str']} (confidance : {j['score']:.4f})')

pred 1: white (confidance : 0.1773)
pred 2: guest (confidance : 0.0885)
pred 3: same (confidance : 0.0624)
pred 4: presidential (confidance : 0.0334)
pred 5: glass (confidance : 0.0263)


## 5. Pre-training ki "Problems" (The Dark Side)

* Bias: Agar internet par galat baatein likhi hain, to model bhi wahi seekh lega.

* Cost: Ek bade model (jaise Llama 3 ya GPT-4) ko pre-train karne mein karodon dollars ki bijli aur GPUs lagte hain.

* No Safety: Pre-trained model "badtameez" ho sakta hai kyunki usne internet ka har tarah ka kachra (toxic text) bhi padha hota hai. (Isi liye Fine-tuning aur RLHF ki zaroorat padti hai).

**Key Takeaway**

Pre-training se model "Grammar" aur "General Knowledge" seekhta hai. Ye ek "Raw Brain" taiyar karne jaisa hai.

## Fine-tuning

Pre-training ne model ko "ABCD" aur "General baatein" sikhayi, lekin Fine-tuning use "Doctor" ya "Lawyer" banati hai.


---





## 1. Fine-tuning ka Concept (The Analogy)
Ek student ne school pass kar liya hai (Pre-training), use basic English/Maths aati hai. Ab wo Medical College jata hai taaki wo sirf bimariyon aur dawaiyon ke baare mein gehraai se seekh sake (Fine-tuning).

**Fine-tuning mein hum 2 cheezein karte hain:**

* Model: Ek pre-trained model lete hain (jaise BERT ya GPT-2).

* Dataset: Ek chhota lekin high-quality expert dataset (e.g., Medical Q&A).

* Training: Model ke weights ko thoda aur adjust karte hain taaki wo us specific field ki bhasha seekh jaye.

## 2. Google Colab Implementation

In [ ]:
!pip install transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [ ]:
small_train_dataset = dataset["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = dataset["test"].shuffle(seed=42).select(range(1000))

In [ ]:
dataset["train"][0]

**Tokenization**

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

In [ ]:
def tokenizer_function(example):
  return tokenizer(example['text'],padding = 'max_length', truncation = True)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_train = small_train_dataset.map(tokenize_function, batched=True)
tokenized_test = small_eval_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

**Fine-tuning**

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. Model Load karein
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

# 2. Training Arguments (Yahan humne 'eval_strategy' use kiya hai)
training_args = TrainingArguments(
    output_dir="test_trainer",
    eval_strategy="epoch"  # <-- Yahan change kiya hai
)

# 3. Trainer Setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

# 4. Train
trainer.train()

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.1

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
1,No log,0.551899


In [ ]:

from transformers import pipeline


classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)


test_text = "This movie was an absolute masterpiece, the acting was top-notch!"
result = classifier(test_text)

print(f"Text: {test_text}")
print(f"Result: {result}")